# Dwarf Example 03: Corpus Overview and Survey Breakdown

**EPS Research — Dwarf/Irregular HI Corpus v1.0**

129 dwarf and irregular galaxies from 4 Local Volume surveys.
This example characterizes the full sample using the flat CSV.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20320362  
**Sources:** LVHIS (Koribalski 2019), VLA-ANGST (Ott 2012), LITTLE THINGS (Oh 2015), WALLABY DR2  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: download corpus from Zenodo ──────────────────
import os, urllib.request
CORPORA = {
    'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
}
for filename, url in CORPORA.items():
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
        print(f"  ✓ {filename}")
    else:
        print(f"  Already present: {filename}")
print("Ready.")


In [ ]:
import csv
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

rows = []
with open('dwarf_irregular_corpus_v1_flat.csv') as f:
    for r in csv.DictReader(f):
        rows.append(r)

surveys = Counter(r['survey'] for r in rows)
tiers   = Counter(r['quality_tier'] for r in rows)
omega_r = sum(1 for r in rows if r.get('omega_ready') == 'True')

print(f"Total galaxies: {len(rows)}")
print("\nSurvey breakdown:")
for s, n in sorted(surveys.items(), key=lambda x: -x[1]):
    print(f"  {s:<15} {n:>4}")
print(f"\nQuality tiers: {dict(tiers)}")
print(f"Omega-ready:   {omega_r}")
print(f"\nDistance range: {min(float(r['distance_mpc']) for r in rows):.1f} -- "
      f"{max(float(r['distance_mpc']) for r in rows):.1f} Mpc")
vmaxs = [float(r['vrot_max_kms']) for r in rows if r.get('vrot_max_kms')]
print(f"Vmax range:     {min(vmaxs):.1f} -- {max(vmaxs):.1f} km/s")


In [ ]:
COLORS = {'LVHIS':'#1f77b4','VLA-ANGST':'#ff7f0e',
          'LITTLE_THINGS':'#2ca02c','WALLABY':'#d62728'}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# Survey pie
labels = list(surveys.keys())
sizes  = [surveys[k] for k in labels]
colors = [COLORS.get(k, '#9467bd') for k in labels]
axes[0].pie(sizes, labels=[f'{l}\n({n})' for l,n in zip(labels,sizes)],
            colors=colors, autopct='%1.0f%%', textprops={'fontsize':8})
axes[0].set_title('Survey Breakdown', fontsize=10)

# Distance histogram
dists = [float(r['distance_mpc']) for r in rows]
axes[1].hist(dists, bins=15, color='#2166ac', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Distance (Mpc)', fontsize=10)
axes[1].set_ylabel('N galaxies', fontsize=10)
axes[1].set_title('Distance Distribution', fontsize=10)

# Vmax histogram
axes[2].hist(vmaxs, bins=15, color='#d62728', alpha=0.8, edgecolor='white')
axes[2].axvline(np.median(vmaxs), color='black', ls='--', lw=1.5,
                label=f'Median={np.median(vmaxs):.0f} km/s')
axes[2].set_xlabel(r'$V_{\rm max}$ (km/s)', fontsize=10)
axes[2].set_ylabel('N galaxies', fontsize=10)
axes[2].set_title('Peak Velocity Distribution', fontsize=10)
axes[2].legend(fontsize=8)

plt.suptitle('EPS Research Dwarf/Irregular Corpus v1.0 — Overview (N=129)',
             fontsize=11)
plt.tight_layout()
plt.savefig('dw03_corpus_overview.png', dpi=150, bbox_inches='tight')
plt.show()